# Chapter 3: Data Understanding

Work through each section below and run every cell yourself. For the full explanations, see the book chapter.

## Table level: how your tables relate

### Confirming a relationship in code

In [1]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.txt", sep=";")
subscriptions = pd.read_csv("../data/raw/subscriptions.txt", sep=";")

# left join keeps every customer, even those with zero matching subscriptions
merged = pd.merge(
    customers[["CustomerID"]],
    subscriptions[["CustomerID", "SubscriptionID"]],
    on="CustomerID",
    how="left",
)

# count how many subscriptions each customer has
subs_per_customer = merged.groupby("CustomerID")["SubscriptionID"].count()
subs_per_customer.describe()

count    1389.000000
mean        5.940965
std         6.657273
min         1.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        50.000000
Name: SubscriptionID, dtype: float64

### Table descriptions

In [2]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1389 entries, 0 to 1388
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   CustomerID  1389 non-null   int64 
 1   Gender      1166 non-null   object
 2   DOB         1389 non-null   object
 3   District    1389 non-null   int64 
 4   ZIP         1389 non-null   int64 
 5   StreeID     1389 non-null   int64 
dtypes: int64(4), object(2)
memory usage: 65.2+ KB


## Data level: getting types and samples right

### Data types

In [3]:
col_types = {
    "CustomerID": "string",
    "Gender": "category",
    "DOB": "string",
    "District": "category",
    "ZIP": "string",
    "StreeID": "string",
}

for col, dtype in col_types.items():
    customers[col] = customers[col].astype(dtype)

customers.dtypes

CustomerID    string[python]
Gender              category
DOB           string[python]
District            category
ZIP           string[python]
StreeID       string[python]
dtype: object

## Variable level: a first statistical read

### Working with dates

In [4]:
date_cols = ["StartDate", "EndDate", "PaymentDate", "RenewalDate"]

# errors="coerce" turns any value that doesn't match the format into a missing value (NaT),
# instead of raising an error and stopping the whole conversion
subscriptions[date_cols] = subscriptions[date_cols].apply(
    lambda col: pd.to_datetime(col, format="%d/%m/%Y", errors="coerce")
)

customers["DOB"] = pd.to_datetime(customers["DOB"], format="%d/%m/%Y", errors="coerce")
subscriptions[date_cols].dtypes

StartDate      datetime64[ns]
EndDate        datetime64[ns]
PaymentDate    datetime64[ns]
RenewalDate    datetime64[ns]
dtype: object

### Describing individual variables

In [5]:
subscriptions["PaymentStatus"].value_counts(dropna=False)

PaymentStatus
Paid        7831
Not Paid     421
Name: count, dtype: int64

In [6]:
subscriptions["TotalPrice"].describe()

count    8091.000000
mean      125.372010
std        95.119942
min         0.000000
25%        29.600000
50%        79.000000
75%       235.000000
max       635.960000
Name: TotalPrice, dtype: float64

In [7]:
print("mean:", subscriptions["TotalPrice"].mean(skipna=True))
print("variance:", subscriptions["TotalPrice"].var(skipna=True))
print("std:", subscriptions["TotalPrice"].std(skipna=True))

mean: 125.3720096403411
variance: 9047.803316850834
std: 95.11994174120815


In [8]:
subscriptions[["TotalPrice", "NbrNewspapers"]].corr().loc["TotalPrice", "NbrNewspapers"]

np.float64(0.898895317946639)

In [9]:
subscriptions.select_dtypes(include="number").describe()

,SubscriptionID,CustomerID,ProductID,Pattern,NbrNewspapers,NbrStart,FormulaID,GrossFormulaPrice,NetFormulaPrice,NetNewspaperPrice,ProductDiscount,FormulaDiscount,TotalDiscount,TotalPrice,TotalCredit
count,8.252000e+03,8.252000e+03,8252.000000,8.252000e+03,8252.000000,8231.000000,8252.000000,8091.000000,8091.000000,8091.000000,8091.000000,8091.000000,8091.000000,8091.000000,8091.000000
mean,5.951908e+05,4.646909e+05,6.249758,1.098604e+06,159.824285,14.995748,3037.575860,135.877842,133.336547,0.856182,2.459114,2.541296,5.000410,125.372010,-0.529440
std,3.359795e+05,3.220835e+05,2.319479,1.162320e+05,117.965557,7.290724,3337.374219,92.596151,93.253428,0.201169,24.595209,10.725060,26.597951,95.119942,5.009556
min,5.395700e+04,4.157000e+03,1.000000,1.000000e+01,2.000000,0.000000,33.000000,0.169733,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-235.000100
25%,3.100938e+05,1.209800e+05,5.000000,1.111110e+06,76.000000,10.000000,875.000000,68.000000,68.000000,0.809211,0.000000,0.000000,0.000000,29.600000,0.000000
50%,5.713405e+05,5.018730e+05,8.000000,1.111110e+06,77.000000,10.000000,918.000000,79.000000,79.000000,0.878289,0.000000,0.000000,0.000000,79.000000,0.000000
75%,8.621750e+05,6.885480e+05,8.000000,1.111110e+06,304.000000,25.000000,4926.000000,235.000000,235.000000,0.973684,0.000000,0.000000,0.000000,235.000000,0.000000
max,1.349168e+06,1.211742e+06,8.000000,1.111110e+06,354.000000,30.000000,17240.000000,310.914306,276.000000,1.120000,267.000000,138.000000,267.000000,635.960000,0.000000


## Saving your progress

In [10]:
subscriptions.to_parquet("../data/raw/subscriptions.parquet", index=False)
customers.to_parquet("../data/raw/customers.parquet", index=False)